# librt smoke tests

`librt` is the mypyc runtime support library. It ships compiled extensions: `base64`, `internal`, `strings`.

In [ ]:
# Example 1: librt.base64 submodule imports and exposes expected encoder/decoder surface.
from librt import base64 as librt_base64

public_names = [n for n in dir(librt_base64) if not n.startswith('_')]
print(f"librt.base64 public names ({len(public_names)}):")
for name in public_names:
    print(f"  {name}: {type(getattr(librt_base64, name)).__name__}")

assert len(public_names) > 0, "librt.base64 should expose at least one public symbol"

In [ ]:
# Example 2: librt.strings submodule imports and exposes its public surface.
from librt import strings as librt_strings

public_names = [n for n in dir(librt_strings) if not n.startswith('_')]
print(f"librt.strings public names ({len(public_names)}):")
for name in public_names:
    obj = getattr(librt_strings, name)
    print(f"  {name}: {type(obj).__name__}")

assert len(public_names) > 0, "librt.strings should expose at least one public symbol"

In [ ]:
# Example 3: librt.internal submodule (mypyc internal helpers) imports.
from librt import internal as librt_internal

public_names = [n for n in dir(librt_internal) if not n.startswith('_')]
print(f"librt.internal public names ({len(public_names)}):")
for name in public_names[:20]:
    obj = getattr(librt_internal, name)
    print(f"  {name}: {type(obj).__name__}")

# internal is allowed to be sparse; just assert it loaded as a module
import types
assert isinstance(librt_internal, types.ModuleType)

In [ ]:
# Example 4: Each submodule is a compiled C-extension (.so), not pure-Python.
import librt
from librt import base64 as librt_base64
from librt import internal as librt_internal
from librt import strings as librt_strings

for mod in (librt_base64, librt_internal, librt_strings):
    file = getattr(mod, '__file__', None)
    print(f"{mod.__name__}: {file}")
    assert file is not None, f"{mod.__name__} should have __file__"
    assert file.endswith('.so'), f"{mod.__name__} should be a compiled .so extension, got {file}"

In [ ]:
# Example 5: Package is correctly typed (PEP 561) — py.typed marker present and .pyi stubs ship.
import os

import librt

pkg_dir = os.path.dirname(librt.__file__) if hasattr(librt, '__file__') and librt.__file__ else librt.__path__[0]
print(f"librt package dir: {pkg_dir}")

contents = sorted(os.listdir(pkg_dir))
print(f"contents: {contents}")

py_typed = os.path.join(pkg_dir, 'py.typed')
has_py_typed = os.path.exists(py_typed)
pyi_files = [f for f in contents if f.endswith('.pyi')]
print(f"py.typed marker present: {has_py_typed}")
print(f".pyi stub files: {pyi_files}")

assert has_py_typed, "librt should ship a py.typed marker (PEP 561)"
assert len(pyi_files) >= 3, f"librt should ship at least 3 .pyi stubs, found {pyi_files}"